# Sunspot Forecasting — Optimization

This notebook explores model enhancements beyond the baseline hybrid:

1. **Prediction Intervals (Quantile Regression)** — calibrated uncertainty bands using LightGBM's native quantile objective
2. **Hyperparameter Tuning (Optuna)** — Bayesian search over LightGBM + Ridge params to directly minimise walk-forward RMSE

**Prerequisites:** Run `00-EDA.ipynb` first (downloads data). No dependency on exported `models/`.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Robust project root detection — works regardless of kernel working directory
_root = os.getcwd()
for _ in range(4):
    if os.path.exists(os.path.join(_root, 'config.yaml')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

os.makedirs('reports', exist_ok=True)
print(f"Working directory: {os.getcwd()}")

from src.utils import load_config
from src.data import load_data
from src.features import create_features, prepare_target
from src.train import train_evaluate, expanding_walk_forward_splits
from src.model import HybridEVTModel, evt_tail_correction

config = load_config('config.yaml')
df = load_data(config['data']['url'], save_path=config['data']['save_path'])

start_period = '1965-01-01'
df_filtered = df.loc[start_period:].copy()
print(f"Period: {df_filtered.index.min().date()} → {df_filtered.index.max().date()}")
print(f"Total days: {len(df_filtered):,}")

In [ ]:
df_feat = create_features(df_filtered, config['features']['lags'], config['features']['rolling_windows'])
df_feat = prepare_target(df_feat, shift=config['features']['target_shift'])

X = df_feat.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y = df_feat['target']

---
## 1. Prediction Intervals via Quantile Regression

The main hybrid model returns **point estimates only**. Here we add calibrated uncertainty bands using LightGBM's native `objective='quantile'`.

We train three models per fold:
- **Lower bound** (α = 0.10)
- **Median** (α = 0.50) — acts as the point forecast
- **Upper bound** (α = 0.90)

This gives an **80% prediction interval**: we expect ~80% of actual values to fall within the band.

Two things to evaluate:
- **Coverage** — does the interval actually contain ~80% of actuals?
- **Width** — how wide are the intervals on average? (tighter = more useful)

In [ ]:
LOWER_Q = 0.10
UPPER_Q = 0.90

lgb_params = dict(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    verbosity=-1,
    n_jobs=-1,
)

lower_preds, median_preds, upper_preds, actuals_all = [], [], [], []

for i, (Xtr, ytr, Xval, yval) in enumerate(expanding_walk_forward_splits(
    X, y,
    config['evaluation']['initial_train_size'],
    config['evaluation']['val_size'],
    config['evaluation']['step_size'],
)):
    if len(Xtr) < 365:
        continue
    if i % 10 == 0:
        print(f"  fold {i:>3} | train={len(Xtr):,}  val={len(Xval)}")

    m_lower  = LGBMRegressor(objective='quantile', alpha=LOWER_Q,  **lgb_params)
    m_median = LGBMRegressor(objective='quantile', alpha=0.50,     **lgb_params)
    m_upper  = LGBMRegressor(objective='quantile', alpha=UPPER_Q,  **lgb_params)

    m_lower.fit(Xtr, ytr)
    m_median.fit(Xtr, ytr)
    m_upper.fit(Xtr, ytr)

    lower  = np.expm1(m_lower.predict(Xval))
    median = np.expm1(m_median.predict(Xval))
    upper  = np.expm1(m_upper.predict(Xval))
    actual = np.expm1(yval.values)

    # Ensure lower <= upper (quantile crossing can occur)
    lower = np.minimum(lower, median)
    upper = np.maximum(upper, median)

    lower_preds.extend(lower)
    median_preds.extend(median)
    upper_preds.extend(upper)
    actuals_all.extend(actual)

lower_preds  = np.array(lower_preds)
median_preds = np.array(median_preds)
upper_preds  = np.array(upper_preds)
actuals_all  = np.array(actuals_all)

print(f"\nTotal validation points: {len(actuals_all):,}")

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
in_interval = (actuals_all >= lower_preds) & (actuals_all <= upper_preds)
coverage    = in_interval.mean() * 100
mean_width  = (upper_preds - lower_preds).mean()
median_rmse = np.sqrt(mean_squared_error(actuals_all, median_preds))
median_mae  = mean_absolute_error(actuals_all, median_preds)

print(f"Quantile regression (80% interval)")
print(f"  Median RMSE : {median_rmse:.2f}")
print(f"  Median MAE  : {median_mae:.2f}")
print(f"  Coverage    : {coverage:.1f}%  (target: 80%)")
print(f"  Mean width  : {mean_width:.1f} sunspots")

In [ ]:
# ── Plot: last 500 validation points with interval band ───────────────────────
n_show = 500
idx = np.arange(len(actuals_all) - n_show, len(actuals_all))

fig, ax = plt.subplots(figsize=(15, 5))
ax.fill_between(idx, lower_preds[idx], upper_preds[idx],
                alpha=0.25, color='steelblue', label='80% interval')
ax.plot(idx, actuals_all[idx],  color='black',     lw=0.8, alpha=0.7, label='Actual')
ax.plot(idx, median_preds[idx], color='steelblue', lw=1.2, linestyle='--', label='Median forecast')
ax.set_title(f'Prediction Intervals — last {n_show} validation points  |  '
             f'Coverage: {coverage:.1f}%  |  Mean width: {mean_width:.1f}')
ax.set_xlabel('Validation index')
ax.set_ylabel('Sunspot count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig('reports/08_prediction_intervals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot: interval width distribution ─────────────────────────────────────────
widths = upper_preds - lower_preds

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(widths, bins=60, color='steelblue', edgecolor='white')
ax.axvline(mean_width, color='red', linestyle='--', label=f'Mean: {mean_width:.1f}')
ax.set_title('Distribution of Interval Widths')
ax.set_xlabel('Width (sunspots)')
ax.set_ylabel('Frequency')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig('reports/09_interval_widths.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Width percentiles:")
for p in [25, 50, 75, 90]:
    print(f"  p{p}: {np.percentile(widths, p):.1f}")

---
## 2. Hyperparameter Tuning with Optuna

The hybrid model's LightGBM params (`num_leaves`, `learning_rate`, `n_estimators`, etc.) and `ridge_alpha` were set by hand. Bayesian optimisation can find a better combination systematically.

**Strategy:** run Optuna against a **fixed holdout split** (last 2 years as validation) rather than the full walk-forward — this keeps each trial fast while still using real out-of-sample data. After finding the best params, we do a single full walk-forward evaluation to get the definitive RMSE.

Baseline to beat: **RMSE 19.24** (from `01-Analysis_and_Modeling.ipynb`).

In [ ]:
# ── Fixed holdout split for Optuna trials ─────────────────────────────────────
CUTOFF = df_filtered.index.max() - pd.DateOffset(years=2)

X_opt_tr = X[X.index <= CUTOFF]
y_opt_tr = y[y.index <= CUTOFF]
X_opt_val = X[X.index > CUTOFF]
y_opt_val = y[y.index > CUTOFF]

print(f"Optuna split — train: {len(X_opt_tr):,}  val: {len(X_opt_val):,}")

In [ ]:
def objective(trial):
    lgb_params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 600),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'verbosity': -1,
        'n_jobs': -1,
    }
    ridge_alpha = trial.suggest_float('ridge_alpha', 0.01, 100.0, log=True)

    model = HybridEVTModel(
        ridge_alpha=ridge_alpha,
        resid_model_type='lgb',
        resid_params=lgb_params,
        max_lag_resid=config['features']['max_lag_resid'],
    )
    preds  = model.fit_predict_val(X_opt_tr, y_opt_tr, X_opt_val)
    actual = np.expm1(y_opt_val.values)
    preds  = evt_tail_correction(preds, actual)
    return np.sqrt(mean_squared_error(actual, preds))


study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=7))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest holdout RMSE : {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Full walk-forward evaluation with best params ─────────────────────────────
import copy

best = study.best_params
config_tuned = copy.deepcopy(config)
config_tuned['model']['ridge_alpha'] = best.pop('ridge_alpha')
config_tuned['model']['lgb_params'] = {**best, 'verbosity': -1, 'n_jobs': -1}

print("Running full walk-forward with tuned params...")
results_tuned = train_evaluate(X, y, config_tuned)

baseline_rmse = 19.24
baseline_mae  = 17.54
tuned_rmse    = results_tuned['hybrid_rmse']
tuned_mae     = results_tuned['hybrid_mae']

print(f"\n{'Model':<25} {'RMSE':>8} {'MAE':>8}")
print('─' * 44)
print(f"{'Baseline (hand-tuned)':<25} {baseline_rmse:>8.2f} {baseline_mae:>8.2f}")
print(f"{'Optuna-tuned':<25} {tuned_rmse:>8.2f} {tuned_mae:>8.2f}")
print()
delta_r = (baseline_rmse - tuned_rmse) / baseline_rmse * 100
delta_m = (baseline_mae  - tuned_mae)  / baseline_mae  * 100
print(f"RMSE change: {delta_r:+.1f}%   MAE change: {delta_m:+.1f}%")

In [ ]:
# ── Optuna visualisation — optimisation history ───────────────────────────────
trials_df = study.trials_dataframe()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.scatter(trials_df.index, trials_df['value'], alpha=0.5, s=20, color='steelblue')
ax1.plot(trials_df.index,
         trials_df['value'].cummin(),
         color='red', lw=1.5, label='Best so far')
ax1.axhline(baseline_rmse, color='gray', linestyle='--', label=f'Baseline {baseline_rmse}')
ax1.set_title('Optimisation History')
ax1.set_xlabel('Trial')
ax1.set_ylabel('Holdout RMSE')
ax1.legend()
ax1.grid(alpha=0.3)

importances = optuna.importance.get_param_importances(study)
ax2.barh(list(importances.keys()), list(importances.values()), color='steelblue')
ax2.set_title('Hyperparameter Importances')
ax2.set_xlabel('Importance')
ax2.grid(alpha=0.3)

plt.tight_layout()
fig.savefig('reports/10_optuna_tuning.png', dpi=150, bbox_inches='tight')
plt.show()